Task 1 — Load, Inspect, and Clean

In [6]:
import pandas as pd

df = pd.read_csv('/content/cardekho_dataset.csv')
df


,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,19537,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
15407,19540,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
15408,19541,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
15409,19542,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


In [7]:
# Basic inspection
print("Shape before cleaning:", df.shape)
print("\nInfo:")
print(df.info())

print("\nMissing %:")
print(df.isnull().sum() * 100 / len(df))

Shape before cleaning: (15411, 14)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB
None

Missing %:
Un

In [8]:
# Drop rows where selling_price is missing
df = df.dropna(subset=['selling_price'])

# Clean mileage, engine, max_power
for col in ['mileage', 'engine', 'max_power']:
    df[col] = df[col].astype(str).str.extract('(\d+\.?\d*)')  # extract numeric part
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col].fillna(df[col].median(), inplace=True)

# Remove unrealistic price values
df = df[(df['selling_price'] != 999999999) & (df['selling_price'] >= 10000)]

# Drop duplicates
df = df.drop_duplicates()

print("\nShape after cleaning:", df.shape)


Shape after cleaning: (15411, 14)


<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_6430/1242565708.py:6: SyntaxWarning: invalid escape sequence '\d'
  df[col] = df[col].astype(str).str.extract('(\d+\.?\d*)')  # extract numeric part
/tmp/ipykernel_6430/1242565708.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipykernel_6430/1242565708.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignm

Task 2 — Encode Categorical Features

In [11]:
# Manual → 0, Automatic → 1
df['transmission_type'] = df['transmission_type'].map({'Manual': 0, 'Automatic': 1})
df['transmission_type']

,transmission_type
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
...,...
15406,NaN
15407,NaN
15408,NaN
15409,NaN


In [12]:
df = pd.get_dummies(df, columns=['fuel_type', 'seller_type'], drop_first=True)

# Final columns
print("\nFinal Columns:")
print(df.columns.tolist())


Final Columns:
['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer']


Markdown Answer

Why drop_first=True?
It avoids the dummy variable trap (multicollinearity), where one feature can be predicted from others, causing issues in models like linear regression.

What does a row of all zeros mean?
It represents the dropped reference category (the first category).
Example: If fuel types are Petrol, Diesel, CNG → Petrol is dropped → all zeros = Petro

Task 3 — Split and Compute Baseline MAE

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import numpy as np

# Define X and y
X = df.drop(columns=['selling_price'])
y = df['selling_price']

# Keep only numeric columns
X = X.select_dtypes(include=[np.number])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Baseline prediction (mean)
baseline_pred = [y_train.mean()] * len(y_test)

# MAE
mae = mean_absolute_error(y_test, baseline_pred)

print(f"Baseline MAE: ₹{round(mae)}")

Baseline MAE: ₹468748
